# Similarité entre groupes d’événements dans la base PREVYO

L’objectif de cette analyse est d’identifier des groupes d’événements fortement similaires à partir de la base PREVYO. L’idée n’est pas seulement de produire un score de similarité, mais d’aider à la décision.

Comme l’a souligné le retour du professeur, un score élevé ne signifie pas automatiquement que deux groupes décrivent exactement le même événement. Il peut s’agir :
	•	d’un quasi-doublon
	•	d’un même événement avec des informations complémentaires
	•	d’un événement proche mais distinct
	•	d’un cas contradictoire nécessitant une vérification humaine

Le but final est donc de faire remonter des paires pertinentes, puis d’analyser ce qu’elles ont en commun et ce qu’elles ont de différent.

#  Import des bibliothèques et chargement du JSON

Chargement des données

On commence par charger le fichier JSON contenant les événements extraits par PREVYO.

# Détection des événements sans date exploitable

In [51]:
import json
import pandas as pd
from itertools import combinations

FILE_PATH = "export.events.json"

with open(FILE_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Nombre total d'events :", len(data))

Nombre total d'events : 11948


#  Construction d’un tableau enrichi des événements

Chaque ligne du tableau correspond à un event.
On conserve ici les informations nécessaires à l’analyse de similarité :
	•	le resultAnalyseId
	•	le taskId
	•	la date de création
	•	le type
	•	les champs métier éventuels (domain, subdomain, risk, riskCarac)
	•	le context
	•	le nombre de nœuds et d’arêtes
	•	les labels de nœuds
	•	les types d’arêtes

Cette étape est importante, car la similarité ne doit pas être réduite à un seul critère. Le professeur a insisté sur le fait qu’il faut conserver plusieurs dimensions sémantiques et structurelles.

In [52]:
rows = []

for i, event in enumerate(data):
    created_at = None
    if "createdAt" in event and isinstance(event["createdAt"], dict):
        created_at = event["createdAt"].get("$date")

    nodes = event.get("nodes", [])
    edges = event.get("edges", [])

    node_labels = []
    for node in nodes:
        node_labels.extend(node.get("labels", []))

    edge_types = [edge.get("type") for edge in edges if edge.get("type")]

    rows.append({
        "event_index": i,
        "resultAnalyseId": event.get("resultAnalyseId"),
        "taskId": event.get("taskId"),
        "createdAt": created_at,
        "type": event.get("type", "UNKNOWN"),
        "subdomain": event.get("subdomain"),
        "domain": event.get("domain"),
        "risk": event.get("risk"),
        "riskCarac": event.get("riskCarac"),
        "context": event.get("context", ""),
        "node_count": len(nodes),
        "edge_count": len(edges),
        "node_labels": node_labels,
        "edge_types": edge_types
    })

df_events_rich = pd.DataFrame(rows)
df_events_rich["createdAt"] = pd.to_datetime(df_events_rich["createdAt"], errors="coerce", utc=True)
df_events_rich["day"] = df_events_rich["createdAt"].dt.strftime("%Y-%m-%d")

print(df_events_rich.head())
print("Nombre d'events :", len(df_events_rich))

   event_index           resultAnalyseId  \
0            0  698b35e042a8083a290c11c7   
1            1  698b35e042a8083a290c11c5   
2            2  698b35e042a8083a290c11c5   
3            3  698b35e042a8083a290c11c5   
4            4  698b35e042a8083a290c11c5   

                                 taskId                        createdAt  \
0  1a076139-ea5e-40e2-8d82-01a1f28b8181 2026-02-10 13:43:03.827000+00:00   
1  9c1f0cb5-58ed-44db-b348-1bb5e5cc3c75 2026-02-10 13:43:05.544000+00:00   
2  9c1f0cb5-58ed-44db-b348-1bb5e5cc3c75 2026-02-10 13:43:05.544000+00:00   
3  9c1f0cb5-58ed-44db-b348-1bb5e5cc3c75 2026-02-10 13:43:05.544000+00:00   
4  9c1f0cb5-58ed-44db-b348-1bb5e5cc3c75 2026-02-10 13:43:05.544000+00:00   

                                                type  \
0                                            UNKNOWN   
1  Thing/Abstract/Event/Transfer/TransferOfUnbias...   
2  Thing/Abstract/Event/Transfer/TransferOfUnbias...   
3                         Thing/Abstract/Event/Build  

# Regroupement par resultAnalyseId

Regroupement des événements par resultAnalyseId

Le regroupement par resultAnalyseId permet de passer d’une analyse au niveau des événements individuels à une analyse au niveau des sources. Chaque groupe correspond à un ensemble d’events issus d’une même analyse.

Cette étape est cohérente avec le retour du professeur, qui a rappelé qu’il est plus pertinent de comparer des ensembles de sous-graphes que des événements isolés pris un à un.

In [54]:
df_groups_rich = (
    df_events_rich.groupby("resultAnalyseId")
    .agg(
        taskId=("taskId", "first"),
        first_day=("day", "first"),
        event_count=("event_index", "count"),
        types=("type", lambda x: sorted(set(x.dropna()))),
        subdomains=("subdomain", lambda x: sorted(set([v for v in x.dropna() if v]))),
        domains=("domain", lambda x: sorted(set([v for v in x.dropna() if v]))),
        risks=("risk", lambda x: sorted(set([v for v in x.dropna() if v]))),
        riskCaracs=("riskCarac", lambda x: sorted(set([v for v in x.dropna() if v]))),
        total_nodes=("node_count", "sum"),
        total_edges=("edge_count", "sum"),
        node_labels=("node_labels", lambda x: sorted(set(label for sublist in x for label in sublist))),
        edge_types=("edge_types", lambda x: sorted(set(edge for sublist in x for edge in sublist)))
    )
    .reset_index()
)

print(df_groups_rich.head())
print("Nombre de groupes :", len(df_groups_rich))

            resultAnalyseId                                taskId   first_day  \
0  698b35e042a8083a290c11c5  9c1f0cb5-58ed-44db-b348-1bb5e5cc3c75  2026-02-10   
1  698b35e042a8083a290c11c6  275e5b36-6790-492c-8242-93c4bb591b37  2026-02-10   
2  698b35e042a8083a290c11c7  1a076139-ea5e-40e2-8d82-01a1f28b8181  2026-02-10   
3  698b35e042a8083a290c11c8  a121f210-4da5-4d54-bef9-2fd9a3295b07  2026-02-10   
4  698b35e042a8083a290c11c9  9fe6d5b0-12a5-43fa-a77c-98de3fccc4a6  2026-02-10   

   event_count                                              types  \
0            4  [Thing/Abstract/Event/Addition, Thing/Abstract...   
1            4  [Thing/Abstract/Event/Addition, Thing/Abstract...   
2            1                                          [UNKNOWN]   
3            5  [Thing/Abstract/Event/Embody, Thing/Abstract/E...   
4            4  [Thing/Abstract/Event, Thing/Abstract/Event/Co...   

                        subdomains                        domains  \
0                            

# Distribution du nombre d’événements par groupe

Distribution du nombre d’événements par groupe

Le professeur a remarqué qu’une moyenne seule ne suffisait pas. Il est donc utile d’observer la distribution du nombre d’événements par resultAnalyseId.

Cela permet de voir si les groupes sont généralement petits, ou si certains articles génèrent un grand nombre de sous-graphes.

In [55]:
group_size_distribution = df_groups_rich["event_count"].value_counts().sort_index()

print(group_size_distribution)
print("Moyenne :", round(df_groups_rich["event_count"].mean(), 2))
print("Médiane :", df_groups_rich["event_count"].median())
print("Maximum :", df_groups_rich["event_count"].max())

event_count
1      82
2     208
3     309
4     400
5     370
6     292
7     232
8     124
9     102
10     65
11     29
12     25
13     10
14     12
15      4
16      3
17      1
18      2
19      2
21      1
Name: count, dtype: int64
Moyenne : 5.26
Médiane : 5.0
Maximum : 21


# Fonction de similarité

Définition d’un score de similarité entre groupes

Le score de similarité combine plusieurs critères sémantiques et structurels :

- types d’événements
- domaines
- sous-domaines
- risques
- caractéristiques de risque
- labels de nœuds
- types d’arêtes
- proximité de taille
- proximité temporelle

Ce score sert à faire remonter des paires candidates. Il ne constitue pas à lui seul une décision de fusion.

#  Calcul du score de similarité entre groupes

Calcul du score de similarité entre groupes

On compare ici les groupes définis par resultAnalyseId.
Chaque paire reçoit un score fondé sur les critères précédents.

Le score maximum théorique est de 17.

In [66]:
results = []

group_records = df_groups_rich.to_dict("records")

for g1, g2 in combinations(group_records, 2):
    type_sim = jaccard(g1["types"], g2["types"])
    domain_sim = jaccard(g1["domains"], g2["domains"])
    subdomain_sim = jaccard(g1["subdomains"], g2["subdomains"])
    risk_sim = jaccard(g1["risks"], g2["risks"])
    riskCarac_sim = jaccard(g1["riskCaracs"], g2["riskCaracs"])
    node_label_sim = jaccard(g1["node_labels"], g2["node_labels"])
    edge_type_sim = jaccard(g1["edge_types"], g2["edge_types"])

    size_close = 1 if abs(g1["event_count"] - g2["event_count"]) <= 1 else 0
    day_same = 1 if g1["first_day"] == g2["first_day"] else 0

    raw_score = (
        3 * type_sim +
        2 * domain_sim +
        2 * subdomain_sim +
        2 * risk_sim +
        2 * riskCarac_sim +
        2 * node_label_sim +
        2 * edge_type_sim +
        size_close +
        day_same
    )

    score_01 = raw_score / 17

    if score_01 >= 0.3:
        results.append({
            "resultAnalyseId_1": g1["resultAnalyseId"],
            "resultAnalyseId_2": g2["resultAnalyseId"],
            "raw_score": round(raw_score, 2),
            "score_01": round(score_01, 3),
            "type_sim": round(type_sim, 2),
            "domain_sim": round(domain_sim, 2),
            "subdomain_sim": round(subdomain_sim, 2),
            "risk_sim": round(risk_sim, 2),
            "riskCarac_sim": round(riskCarac_sim, 2),
            "node_label_sim": round(node_label_sim, 2),
            "edge_type_sim": round(edge_type_sim, 2),
            "event_count_1": g1["event_count"],
            "event_count_2": g2["event_count"],
            "day_same": day_same
        })

df_group_similarity_norm = pd.DataFrame(results).sort_values("score_01", ascending=False)

print(df_group_similarity_norm.head(20))
print("Nombre de paires retenues :", len(df_group_similarity_norm))

              resultAnalyseId_1         resultAnalyseId_2  raw_score  \
16267  6991a0987eb36c285af0a947  6991f2c17eb36c285af0a970      17.00   
16053  699168567eb36c285af0a92e  6991a0987eb36c285af0a947      16.00   
16057  699168567eb36c285af0a92e  6991f2c17eb36c285af0a970      16.00   
22564  699af4bbbf23573ae1689d53  699b2aa5bf23573ae1689d6e      15.00   
1428   698b361542a8083a290c11f6  698b363242a8083a290c1216      15.00   
5036   698b385642a8083a290c1234  698b385742a8083a290c1236      15.00   
16518  6991e9617eb36c285af0a96d  699223fb7eb36c285af0a983      15.00   
1322   698b361442a8083a290c11f5  698b363242a8083a290c1216      15.00   
1292   698b361442a8083a290c11f5  698b361542a8083a290c11f6      15.00   
4894   698b384f42a8083a290c122f  698b385742a8083a290c1236      15.00   
4892   698b384f42a8083a290c122f  698b385642a8083a290c1234      15.00   
17316  6992edce7eb36c285af0a9c4  69930c477eb36c285af0a9d9      15.00   
4164   698b383a42a8083a290c121d  698b384842a8083a290c1227      1

# Filtrage des paires très fortement similaires

Sélection des paires très fortement similaires

Afin de réduire le bruit, on conserve uniquement les paires les plus fortes, c’est-à-dire celles ayant :
	•	un score élevé
	•	une proximité parfaite ou quasi parfaite sur les types
	•	une forte proximité sur les labels de nœuds
	•	une forte proximité sur les types d’arêtes

Ces paires constituent de bons candidats à une analyse qualitative plus fine.

In [67]:
top_similar_groups = df_group_similarity[
    (df_group_similarity["score"] >= 14) &
    (df_group_similarity["type_sim"] == 1.0) &
    (df_group_similarity["node_label_sim"] >= 0.9) &
    (df_group_similarity["edge_type_sim"] >= 0.8)
].copy()

print(top_similar_groups.head(20))
print("Nombre de paires très fortement similaires :", len(top_similar_groups))

              resultAnalyseId_1         resultAnalyseId_2  score  day_same  \
17957  6991a0987eb36c285af0a947  6991f2c17eb36c285af0a970   17.0      True   
5028   698b384e42a8083a290c122c  698b385e42a8083a290c123e   15.0      True   
5117   698b384f42a8083a290c122f  698b385642a8083a290c1234   15.0      True   
24986  699ad623bf23573ae1689d3d  699adf84bf23573ae1689d47   15.0      True   
18851  6992b7c67eb36c285af0a9ac  69930c477eb36c285af0a9d9   15.0      True   
18850  6992b7c67eb36c285af0a9ac  6992edce7eb36c285af0a9c4   15.0      True   
5119   698b384f42a8083a290c122f  698b385742a8083a290c1236   15.0      True   
25090  699af4bbbf23573ae1689d53  699b2aa5bf23573ae1689d6e   15.0      True   
19143  6992edce7eb36c285af0a9c4  69930c477eb36c285af0a9d9   15.0      True   
1399   698b361442a8083a290c11f5  698b361542a8083a290c11f6   15.0      True   
5264   698b385642a8083a290c1234  698b385742a8083a290c1236   15.0      True   
1541   698b361542a8083a290c11f6  698b363242a8083a290c1216   15.0

# Exemple 1, cas de redondance forte

Exemple 1 : cas de redondance forte

Le premier exemple illustre un cas où les deux groupes présentent des extraits locaux quasiment identiques. On est ici proche du quasi-doublon.

In [68]:
example_1 = "698b384f42a8083a290c122f"
example_2 = "698b385642a8083a290c1234"

for rid in [example_1, example_2]:
    print(f"\nresultAnalyseId : {rid}\n")
    subset = df_events_rich[
        df_events_rich["resultAnalyseId"] == rid
    ][["event_index", "context"]]
    print(subset.to_string(index=False))


resultAnalyseId : 698b384f42a8083a290c122f

 event_index                                                                                                                                                                                                       context
         468                        Le 14 novembre 2025, l’exploitant de la centrale nucléaire de Gravelines a déclaré à l’Autorité de sûreté nucléaire et de radioprotection un événement significatif relatif à l’indisp
         469 entrale nucléaire de Gravelines a déclaré à l’Autorité de sûreté nucléaire et de radioprotection un événement significatif relatif à l’indisponibilité de la ligne de décharge du circuit primaire principal.

resultAnalyseId : 698b385642a8083a290c1234

 event_index                                                                                                                                                                                                 context
         505                            

# Exemple 2, cas de similarité forte sans fusion automatique

Exemple 2 : similarité structurelle forte sans identité événementielle stricte

Le deuxième exemple illustre un cas plus subtil. Les deux groupes ont des profils très proches, mais ne décrivent pas exactement le même incident.

In [70]:
example_1 = "6991a0987eb36c285af0a947"
example_2 = "6991f2c17eb36c285af0a970"

pair_details = df_groups_rich[
    df_groups_rich["resultAnalyseId"].isin([example_1, example_2])
][[
    "resultAnalyseId",
    "first_day",
    "event_count",
    "types",
    "subdomains",
    "domains",
    "risks",
    "riskCaracs",
    "node_labels",
    "edge_types"
]]

for _, row in pair_details.iterrows():
    rid = row["resultAnalyseId"]

    print(f"\n{'='*100}")
    print(f"resultAnalyseId : {rid}")
    print(f"Jour principal : {row['first_day']}")
    print(f"Nombre d'events : {row['event_count']}")
    print("Types :", ", ".join(row["types"]))
    print("Subdomains :", ", ".join(row["subdomains"]) if row["subdomains"] else "Aucun")
    print("Domains :", ", ".join(row["domains"]) if row["domains"] else "Aucun")
    print("Risks :", ", ".join(row["risks"]) if row["risks"] else "Aucun")
    print("RiskCaracs :", ", ".join(row["riskCaracs"]) if row["riskCaracs"] else "Aucun")
    print("Node labels :", ", ".join(row["node_labels"]))
    print("Edge types :", ", ".join(row["edge_types"]))

    print("\nContexts associés :")
    contexts = df_events_rich[df_events_rich["resultAnalyseId"] == rid][["event_index", "context"]]
    for _, ctx_row in contexts.iterrows():
        print(f"- event_index {ctx_row['event_index']} : {ctx_row['context']}")


resultAnalyseId : 6991a0987eb36c285af0a947
Jour principal : 2026-02-15
Nombre d'events : 5
Types : Thing/Abstract/Event/Alternative, Thing/Abstract/Event/NaturalEvents/WaterOverflow, Thing/Abstract/Event/Transfer/TransferOfUnbiasedInformation, Thing/Abstract/Event/Want
Subdomains : Thing/Abstract/Domain/DrinkingWater
Domains : Thing/Abstract/Organization/Operator/WaterManagement
Risks : Thing/Abstract/Risk/Natural
RiskCaracs : Thing/Abstract/Event/NaturalEvents/Thunderstorm
Node labels : Thing, Thing/Abstract/Color, Thing/Abstract/Event/Alternative, Thing/Abstract/Event/NaturalEvents/WaterOverflow, Thing/Abstract/Event/Transfer/TransferOfUnbiasedInformation, Thing/Abstract/Event/Want, Thing/Abstract/Time
Edge types : Addition, Alternative, Time, TimeMin

Contexts associés :
- event_index 5066 : s cours d’eau en vigilance orange ou rouge, des débordements importants et majeurs sont en cours ou attendus dans les prochaines vingt-quatre heures , prévient Vigicrues.
- event_index 5067 : o

Cette paire présente une similarité structurelle très forte. Les deux groupes partagent le même jour principal, les mêmes types d’événements, le même sous-domaine nucléaire, le même domaine énergie, le même risque industriel, ainsi que les mêmes labels de nœuds et types d’arêtes.

Cependant, une lecture plus qualitative montre qu’ils ne décrivent pas exactement le même incident. Le premier groupe concerne une indisponibilité du système de contournement de la turbine, tandis que le second concerne une indisponibilité du système d’aspersion.

Cette paire illustre donc bien la limite du score : une forte proximité structurelle ne signifie pas nécessairement identité événementielle. Dans ce cas, il ne faut pas conclure à une fusion automatique.